Instrukcja uruchomienia:

1. w katalogu milvus_db : docker compose up -d    (docker ps , żeby sprawdzić czy działa)

2. w główbnym katalogu source .venv/bin/activate
3. uv sync


In [121]:
import ollama
from pymilvus import MilvusClient, FieldSchema, DataType, CollectionSchema

LLM_MODEL = "gemma2:2b"
EMBEDDING_MODEL = "nomic-embed-text"

VECTOR_LENGTH = 768

host = "localhost"
port = "19530"

COLLECTION_NAME = "csr_demographics_single_record_demo"


In [122]:
DEMO_RECORD = {
    "cell_id": "T14_1_1_R01_C01",
    "section": "Demographics",
    "table_id": "14.1.1",
    "variable": "Age",
    "arm": "Placebo",
    "text": "Age in Placebo arm: n=50, mean=56.5 years, SD=8.2, median=55.0, range=42-72."
}

In [123]:
def embed_text(text: str) -> list[float]:
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]

In [124]:
def setup_milvus_collection() -> MilvusClient:
    milvus_client = MilvusClient(host=host, port=port)

    if milvus_client.has_collection(COLLECTION_NAME):
        milvus_client.drop_collection(COLLECTION_NAME)

    id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True)
    text_field = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096)
    embedding_field = FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH)
    section_field = FieldSchema(name="section", dtype=DataType.VARCHAR, max_length=128)
    table_id_field = FieldSchema(name="table_id", dtype=DataType.VARCHAR, max_length=64)
    variable_field = FieldSchema(name="variable", dtype=DataType.VARCHAR, max_length=128)
    arm_field = FieldSchema(name="arm", dtype=DataType.VARCHAR, max_length=128)
    cell_id_field = FieldSchema(name="cell_id", dtype=DataType.VARCHAR, max_length=128)


    fields = [id_field, text_field, embedding_field, table_id_field, section_field, variable_field, arm_field, cell_id_field]
    
    schema = CollectionSchema(fields=fields, auto_id=False, enable_dynamic_field=True)

    milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)

    index_params = milvus_client.prepare_index_params()

    index_params.add_index(field_name="embedding", index_type="HNSW", metric_type="L2", params={"M": 4, "efConstruction": 64})

    milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

    milvus_client.load_collection(collection_name=COLLECTION_NAME)

    return milvus_client



In [125]:
def index_record(milvus_client: MilvusClient, record: dict) -> None:
    embedding = embed_text(record["text"])

    entity = {
        "id": 1,
        "text": record["text"],
        "embedding": embedding,
        "table_id": record["table_id"],
        "section": record["section"],
        "variable": record["variable"],
        "arm": record["arm"],
        "cell_id": record["cell_id"]
    }

    milvus_client.insert(collection_name=COLLECTION_NAME, data=[entity])

    milvus_client.flush(collection_name=COLLECTION_NAME)


In [126]:
def retrieve_context(milvus_client: MilvusClient, query: str) -> str:

    query_embedding = embed_text(query)

    results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        anns_field="embedding",
        limit=1,
        filter='section == "Demographics" and table_id == "14.1.1"',
        output_fields=["text", "table_id", "section", "variable", "arm", "cell_id"],
        search_params={"metric_type": "L2", "params": {"ef": 32}}
    )

    entity = results[0][0]["entity"]

    context = f"""
    Source cell_id: {entity["cell_id"]}
    Table: {entity["table_id"]}
    Section: {entity["section"]}
    Variable: {entity["variable"]}
    Arm: {entity["arm"]}
    Verified source data: {entity["text"]}
    """.strip()

    return context

In [127]:
def generate_narrative(context: str) -> str:
    prompt = f"""
    You are a clinical study report medical writing assistant.

    Your task: Generate one concise CSR demographics sentence about age.

    Critical rules:
    - Use only the retrieved context below.
    - Do not invent, estimate, round or modify any number.
    - Every numerical statement must be traceable to the provided cell_id.
    - If a value is not present in the context, do not mention it.
    - Write in formal CSR style.

    Retrieved context:
    {context}
    """

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You generate factual, traceable clinical study report text, using only the provided source context"
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={"temperature": 0}
    )

    return response["message"]["content"]

In [128]:
def main():
    milvus_client = setup_milvus_collection()

    index_record(milvus_client, DEMO_RECORD)

    user_query = "Generate CSR demographics narrative about age in the placebo arm."

    context = retrieve_context(milvus_client=milvus_client, query=user_query)

    print("Retrieved context:")
    print(context)

    narrative = generate_narrative(context)

    print("Generate CSR narrative:")
    print(narrative)


In [129]:
main()

Retrieved context:
Source cell_id: T14_1_1_R01_C01
    Table: 14.1.1
    Section: Demographics
    Variable: Age
    Arm: Placebo
    Verified source data: Age in Placebo arm: n=50, mean=56.5 years, SD=8.2, median=55.0, range=42-72.
Generate CSR narrative:
The mean age of participants in the placebo arm was 56.5 years with a standard deviation of 8.2 years, and a median age of 55.0 years.  

